# 03 · Preparación de los Datos

**Fase CRISP-DM:** 3 de 6 — Data Preparation  
Del CSV crudo a una tabla analítica: una fila por fecha, columnas por embalse, indicadores derivados y cruce con comunicados.

In [1]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent
sys.path.insert(0, str(RAIZ / "src"))

import pandas as pd
from constantes import EMBALSES, ARCHIVO_COTAS_PROCESADAS, ARCHIVO_COMUNICADOS

df = pd.read_csv(RAIZ / "data" / "raw" / "cotas_historico.csv", parse_dates=["fecha"])
df = df.sort_values(["embalse", "fecha"]).reset_index(drop=True)
print(f"{len(df)} mediciones crudas")

4899 mediciones crudas


## 1. Limpieza

1. Eliminar duplicados exactos de (embalse, fecha) quedándonos con la lectura más reciente.
2. Validar tipos y rangos físicos (cota > 0).
3. Ordenar por fecha.

In [2]:
antes = len(df)
df = df.sort_values("fecha_consulta").drop_duplicates(subset=["embalse", "fecha"], keep="last")
df = df[(df["cota_msnm"] > 0) & df["cota_msnm"].notna()]
print(f"Duplicados/nulos eliminados: {antes - len(df)}")

Duplicados/nulos eliminados: 0


## 2. Tabla pivote: una fila por fecha, una columna por embalse

In [3]:
tabla = df.pivot_table(index="fecha", columns="embalse", values="cota_msnm", aggfunc="last")
tabla = tabla.asfreq("D")  # calendario diario completo, NaN donde falta el dato
tabla.head()

embalse,Amaluza,Mazar,Sopladora
fecha,,,
2022-01-01,1984.03,2151.97,1316.19
2022-01-02,1983.34,2151.65,1315.69
2022-01-03,1983.23,2151.06,1315.81
2022-01-04,1983.23,2150.16,1315.75
2022-01-05,1983.21,2149.23,1315.93


## 3. Indicadores derivados

Para cada embalse:
- `cota − cota_min` y `cota_max − cota`: margen a los extremos de la banda normal (metros).
- `cota − cota_critica`: distancia al nivel crítico donde aplique (Mazar).
- `dentro_de_banda`: bandera booleana.

In [4]:
procesado = tabla.copy()
for embalse, conf in EMBALSES.items():
    col = procesado[embalse]
    procesado[f"{embalse}_margen_sobre_min"] = (col - conf["cota_min"]).round(2)
    procesado[f"{embalse}_margen_bajo_max"] = (conf["cota_max"] - col).round(2)
    procesado[f"{embalse}_dentro_de_banda"] = col.between(conf["cota_min"], conf["cota_max"])
    if conf["cota_critica"] is not None:
        procesado[f"{embalse}_distancia_a_critico"] = (col - conf["cota_critica"]).round(2)
procesado.head()

embalse,Amaluza,Mazar,Sopladora,Mazar_margen_sobre_min,Mazar_margen_bajo_max,Mazar_dentro_de_banda,Mazar_distancia_a_critico,Amaluza_margen_sobre_min,Amaluza_margen_bajo_max,Amaluza_dentro_de_banda,Sopladora_margen_sobre_min,Sopladora_margen_bajo_max,Sopladora_dentro_de_banda
fecha,,,,,,,,,,,,,
2022-01-01,1984.03,2151.97,1316.19,53.97,1.03,True,36.97,9.03,6.97,True,4.19,1.81,True
2022-01-02,1983.34,2151.65,1315.69,53.65,1.35,True,36.65,8.34,7.66,True,3.69,2.31,True
2022-01-03,1983.23,2151.06,1315.81,53.06,1.94,True,36.06,8.23,7.77,True,3.81,2.19,True
2022-01-04,1983.23,2150.16,1315.75,52.16,2.84,True,35.16,8.23,7.77,True,3.75,2.25,True
2022-01-05,1983.21,2149.23,1315.93,51.23,3.77,True,34.23,8.21,7.79,True,3.93,2.07,True


## 4. Cruce con comunicados oficiales

Si `data/comunicados/comunicados.csv` tiene registros, a cada comunicado se le asocia la cota medida de ese día (y la de 7 días antes como contexto de tendencia). Si está vacío, el cruce queda listo para cuando haya registros — nada más cambia.

In [5]:
try:
    com = pd.read_csv(ARCHIVO_COMUNICADOS, parse_dates=["fecha"])
    com = com.dropna(subset=["fecha"])
except Exception:
    com = pd.DataFrame()

if len(com):
    # ¿sobre qué embalse habla el comunicado? por ahora lo cruzamos con los tres
    filas = []
    for _, c in com.iterrows():
        fila = {"fecha": c["fecha"], "fuente": c["fuente"], "mensaje": c["mensaje_parafraseado"]}
        for embalse in EMBALSES:
            if c["fecha"] in tabla.index:
                fila[f"cota_{embalse}"] = tabla.at[c["fecha"], embalse]
                idx_7d = c["fecha"] - pd.Timedelta(days=7)
                if idx_7d in tabla.index:
                    fila[f"cota_{embalse}_hace_7d"] = tabla.at[idx_7d, embalse]
        filas.append(fila)
    comunicados_con_cota = pd.DataFrame(filas)
    display(comunicados_con_cota)
else:
    comunicados_con_cota = pd.DataFrame()
    print("comunicados.csv sin registros aún — el cruce se activará cuando se registren.")

comunicados.csv sin registros aún — el cruce se activará cuando se registren.


## 5. Persistencia

In [6]:
ARCHIVO_COTAS_PROCESADAS.parent.mkdir(parents=True, exist_ok=True)
procesado.to_csv(ARCHIVO_COTAS_PROCESADAS, index_label="fecha")
if len(comunicados_con_cota):
    comunicados_con_cota.to_csv(RAIZ / "data" / "processed" / "comunicados_con_cota.csv", index=False)
print(f"→ {ARCHIVO_COTAS_PROCESADAS} ({len(procesado)} filas)")

→ C:\Users\Jordan\.zcode\workspace\default\cotas-embalses-ecuador\data\processed\cotas_diarias.csv (1688 filas)


## 6. Informe de calidad de la preparación

- Filas crudas → filas procesadas (deduplicación).
- Días con dato por embalse tras el reindexado diario.
- Porcentaje de días dentro de banda por embalse (descriptivo, no valorativo).

In [7]:
resumen = []
for embalse in EMBALSES:
    serie = procesado[embalse]
    resumen.append({
        "embalse": embalse,
        "dias_con_dato": int(serie.notna().sum()),
        "dias_sin_dato": int(serie.isna().sum()),
        "pct_dentro_de_banda": round(100 * procesado[f"{embalse}_dentro_de_banda"].mean(), 1),
    })
pd.DataFrame(resumen)

,embalse,dias_con_dato,dias_sin_dato,pct_dentro_de_banda
0,Mazar,1633,55,73.5
1,Amaluza,1633,55,94.3
2,Sopladora,1633,55,96.7
